##  Imports, configuration, and dataset path discovery

In [ ]:
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)


# Locate Dataset A the same way as the exploratory notebook:
# search for a directory whose direct children include ses_*
# session folders, rather than assuming one fixed relative path.


def _has_session_children(directory: Path) -> bool:
    try:
        return any(
            child.is_dir() and child.name.startswith("ses_")
            for child in directory.iterdir()
        )
    except (PermissionError, FileNotFoundError):
        return False


def find_dataset_sessions_root(search_roots, dataset_label, max_depth=3):
    tried = []
    seen = set()
    frontier = [(root, 0) for root in search_roots if root.exists()]
    while frontier:
        current, depth = frontier.pop(0)
        if current in seen:
            continue
        seen.add(current)
        tried.append(current)
        if _has_session_children(current):
            return current
        if depth < max_depth:
            try:
                for child in current.iterdir():
                    if child.is_dir():
                        frontier.append((child, depth + 1))
            except (PermissionError, FileNotFoundError):
                continue
    raise FileNotFoundError(
        f"Could not locate a sessions folder for {dataset_label}.\n"
        f"Checked {len(tried)} candidate directories, including:\n"
        + "\n".join(f"  - {p}" for p in tried[:15])
    )


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # this notebook lives in pipelines/

dataset_a_search_roots = [
    PROJECT_ROOT / "notebooks" / "dataset_A",
    PROJECT_ROOT / "dataset_A",
    NOTEBOOK_DIR / "dataset_A",
]

DATASET_A_DIR = find_dataset_sessions_root(dataset_a_search_roots, "Dataset A")
print("\nDataset A sessions root:", DATASET_A_DIR)


# Artifacts directory structure (matches the existing project layout)


ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
PREPROCESSING_DIR = ARTIFACTS_DIR / "preprocessing"
METADATA_DIR = ARTIFACTS_DIR / "metadata"

for directory in [ARTIFACTS_DIR, MODELS_DIR, PREPROCESSING_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Artifacts directory:", ARTIFACTS_DIR)


pandas version: 3.0.5
numpy version : 2.5.3

Dataset A sessions root: c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\notebooks\dataset_A\dataset_a_combined
Artifacts directory: c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts


##  Load all Dataset A raw events

Loads every `events.jsonl` across all sessions/chunks, chronologically
sorted per session. Same logic validated earlier (63/63 sessions,
162,768/162,768 events matched the reference audit exactly).

In [4]:
def load_session_events(session_dir: Path) -> pd.DataFrame:
    event_files = sorted(session_dir.rglob("events.jsonl"))
    if not event_files:
        raise FileNotFoundError(f"No events.jsonl files found under {session_dir}")

    records = []
    malformed_line_count = 0

    for event_file in event_files:
        chunk_dir_name = event_file.parent.name
        with open(event_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    malformed_line_count += 1
                    continue

                context = event.get("context") or {}
                active_app = context.get("active_app") or {}
                correlation = event.get("correlation") or {}

                records.append({
                    "session_id": session_dir.name,
                    "event_id": event.get("event_id"),
                    "timestamp_iso": event.get("timestamp_iso"),
                    "layer": event.get("layer"),
                    "event_type": event.get("event_type"),
                    "active_app_name": active_app.get("app_name"),
                    "active_window_title": active_app.get("window_title"),
                    "sequence_number": correlation.get("sequence_number"),
                    "source_chunk_dir": chunk_dir_name,
                })

    if malformed_line_count:
        print(f"  WARNING: skipped {malformed_line_count} malformed line(s) in {session_dir.name}")

    events_df = pd.DataFrame.from_records(records)
    events_df["timestamp"] = pd.to_datetime(
        events_df["timestamp_iso"].str.replace("Z", "+00:00", regex=False),
        utc=True,
        errors="coerce",
    )
    events_df = events_df.sort_values(
        by=["timestamp", "sequence_number"], kind="mergesort", na_position="last"
    ).reset_index(drop=True)
    return events_df


dataset_a_sessions = sorted([
    p for p in DATASET_A_DIR.iterdir() if p.is_dir() and p.name.startswith("ses_")
])
print("Sessions found:", len(dataset_a_sessions), "(expected 63)")

_start = time.perf_counter()
_frames = [load_session_events(s) for s in dataset_a_sessions]
ALL_EVENTS_A = pd.concat(_frames, ignore_index=True)
print(f"Loaded {len(ALL_EVENTS_A):,} events in {time.perf_counter() - _start:.1f}s (expected 162,768)")


Sessions found: 63 (expected 63)
Loaded 162,768 events in 11.0s (expected 162,768)


##  Load ground truth and reconstruct segments (resume-aware)

State machine, validated earlier to reconstruct exactly 2,009 segments
(1,819 from `process_started` + 190 from `process_resumed`) with 0
mismatches against the `from`/`current_process` fields:

- `process_started` / `process_resumed` -> close whatever is open, open a new interval.
- `process_switched_out` / `process_suspended` -> close whatever is open, open nothing.
- Anything still open at the end -> closed at `session_ended` (or max GT timestamp).

In [5]:
def load_session_gt(session_dir: Path) -> list:
    gt_file = session_dir / "gt.jsonl"
    if not gt_file.exists():
        raise FileNotFoundError(f"No gt.jsonl found under {session_dir}")

    records = []
    with open(gt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            record["session_id"] = session_dir.name
            records.append(record)

    records.sort(key=lambda r: r.get("ts_utc") or "")
    return records


_gt_records = []
for session_dir in dataset_a_sessions:
    _gt_records.extend(load_session_gt(session_dir))

ALL_GT_A = pd.DataFrame(_gt_records)
print("GT records loaded:", len(ALL_GT_A), "(expected 10,609)")


def reconstruct_gt_segments(session_gt: pd.DataFrame):
    session_gt = session_gt.sort_values("ts_utc").reset_index(drop=True)
    session_id = session_gt["session_id"].iloc[0]

    session_end_rows = session_gt[session_gt["event"] == "session_ended"]
    session_end_ts = (
        session_end_rows["ts_utc"].iloc[0] if not session_end_rows.empty
        else session_gt["ts_utc"].max()
    )

    relevant = session_gt[session_gt["event"].isin(
        ["process_started", "process_switched_out", "process_suspended", "process_resumed"]
    )]

    segments = []
    open_interval = {"active": False}

    def close_open_interval(end_ts, closed_by):
        if open_interval["active"]:
            segments.append({
                "session_id": session_id,
                "case_id": open_interval["case_id"],
                "process_code": open_interval["process_code"],
                "phase": open_interval["phase"],
                "start": open_interval["start_ts"],
                "end": end_ts,
                "closed_by": closed_by,
            })
            open_interval["active"] = False

    for _, row in relevant.iterrows():
        event_type = row["event"]
        ts = row["ts_utc"]

        if event_type in ("process_started", "process_resumed"):
            close_open_interval(ts, closed_by=f"implicit_{event_type}")
            open_interval.update({
                "active": True,
                "case_id": row.get("case_id"),
                "process_code": row.get("process_code") if pd.notna(row.get("process_code")) else row.get("current_process"),
                "phase": int(row["phase"]) if pd.notna(row.get("phase")) else 1,
                "start_ts": ts,
            })
        elif event_type in ("process_switched_out", "process_suspended"):
            close_open_interval(ts, closed_by=event_type)

    close_open_interval(session_end_ts, closed_by="session_end_fallback")
    return segments


_all_segments = []
for session_id, group in ALL_GT_A.groupby("session_id"):
    _all_segments.extend(reconstruct_gt_segments(group))

GT_SEGMENTS_A = pd.DataFrame(_all_segments)
GT_SEGMENTS_A["start"] = pd.to_datetime(GT_SEGMENTS_A["start"], utc=True)
GT_SEGMENTS_A["end"] = pd.to_datetime(GT_SEGMENTS_A["end"], utc=True)
GT_SEGMENTS_A["duration_seconds"] = (GT_SEGMENTS_A["end"] - GT_SEGMENTS_A["start"]).dt.total_seconds()

print("GT segments reconstructed:", len(GT_SEGMENTS_A), "(expected 2,009)")
print("Segments from process_resumed (phase 2):", (GT_SEGMENTS_A["phase"] == 2).sum(), "(expected 190)")
print("Minimum true segment duration (s):", round(GT_SEGMENTS_A["duration_seconds"].min(), 2))


GT records loaded: 10609 (expected 10,609)
GT segments reconstructed: 2009 (expected 2,009)
Segments from process_resumed (phase 2): 190 (expected 190)
Minimum true segment duration (s): 16.64


##  Candidate boundary generation (context-shift events only)

Restricting candidates to context-shift event types was validated to
recover ~84% of true starts within 2s, vs ~62% using `app_switch` alone.

In [6]:
BOUNDARY_EVENT_TYPES = {
    "app_switch",
    "window_title_change",
    "window_state_change",
    "dialog_opened",
    "dialog_closed",
    "browser_navigation",
    "browser_tab_event",
}

CANDIDATES = ALL_EVENTS_A[ALL_EVENTS_A["event_type"].isin(BOUNDARY_EVENT_TYPES)].copy()

n_before_dedup = len(CANDIDATES)
CANDIDATES = CANDIDATES.drop_duplicates(subset=["session_id", "timestamp"], keep="first")
print("Exact-timestamp duplicate candidates removed:", n_before_dedup - len(CANDIDATES))

CANDIDATES = CANDIDATES.sort_values(["session_id", "timestamp"]).reset_index(drop=True)

print("Candidate boundary events:", len(CANDIDATES),
      f"({len(CANDIDATES) / len(ALL_EVENTS_A):.1%} of all raw events)")
print("\nBreakdown by event_type:")
print(CANDIDATES["event_type"].value_counts())


Exact-timestamp duplicate candidates removed: 46
Candidate boundary events: 53109 (32.6% of all raw events)

Breakdown by event_type:
event_type
app_switch             50588
browser_navigation      1682
window_title_change      451
window_state_change      232
browser_tab_event        100
dialog_opened             28
dialog_closed             28
Name: count, dtype: int64


## Sanity check: candidate recall against ground truth

In [7]:
def nearest_gap_ms(sorted_candidate_ns: np.ndarray, target_ns: np.ndarray) -> np.ndarray:
    """Vectorized nearest-neighbor absolute gap (ms) via binary search."""
    if len(sorted_candidate_ns) == 0:
        return np.full(len(target_ns), np.nan)
    idx = np.searchsorted(sorted_candidate_ns, target_ns)
    idx_right = np.clip(idx, 0, len(sorted_candidate_ns) - 1)
    idx_left = np.clip(idx - 1, 0, len(sorted_candidate_ns) - 1)
    gap_right = np.abs(sorted_candidate_ns[idx_right] - target_ns)
    gap_left = np.abs(sorted_candidate_ns[idx_left] - target_ns)
    return np.minimum(gap_left, gap_right) / 1e6


def compute_nearest_gaps(gt_df, candidate_df, ts_col_gt="start", ts_col_cand="timestamp"):
    gap_ms = np.full(len(gt_df), np.nan)
    gt_df = gt_df.reset_index(drop=True)
    for session_id, session_gt in gt_df.groupby("session_id"):
        cand_ns = (
            candidate_df.loc[candidate_df["session_id"] == session_id, ts_col_cand]
            .sort_values().values.astype("datetime64[ns]").astype("int64")
        )
        target_ns = session_gt[ts_col_gt].values.astype("datetime64[ns]").astype("int64")
        gap_ms[session_gt.index.values] = nearest_gap_ms(cand_ns, target_ns)
    return gap_ms


GT_SEGMENTS_A["gap_ms_to_candidate"] = compute_nearest_gaps(GT_SEGMENTS_A, CANDIDATES)

for tol in [500, 1000, 2000, 5000]:
    recall = (GT_SEGMENTS_A["gap_ms_to_candidate"] <= tol).mean()
    print(f"Candidate recall @ {tol}ms: {recall:.1%}")

print("\n(Expected roughly 60%/75%/84%/87% at these tolerances, based on prior analysis.)")


Candidate recall @ 500ms: 60.9%
Candidate recall @ 1000ms: 75.5%
Candidate recall @ 2000ms: 84.2%
Candidate recall @ 5000ms: 87.0%

(Expected roughly 60%/75%/84%/87% at these tolerances, based on prior analysis.)


## Feature engineering (per candidate boundary event)

Features are built only for the ~53k restricted candidates, not all 162k
raw events -- this keeps the classification problem's positive rate and
search space aligned with what we already validated works.

In [ ]:
ALL_EVENTS_A = ALL_EVENTS_A.sort_values(["session_id", "timestamp"]).reset_index(drop=True)
ALL_EVENTS_A["time_since_prev_event_s"] = (
    ALL_EVENTS_A.groupby("session_id")["timestamp"].diff().dt.total_seconds()
)

# --- gap before this candidate, relative to the previous RAW event of any type ---
CANDIDATES = CANDIDATES.merge(
    ALL_EVENTS_A[["event_id", "time_since_prev_event_s"]],
    on="event_id", how="left",
)

CANDIDATES = CANDIDATES.sort_values(["session_id", "timestamp"]).reset_index(drop=True)

# --- gap before this candidate, relative to the previous CANDIDATE ---
CANDIDATES["gap_before_candidate_s"] = (
    CANDIDATES.groupby("session_id")["timestamp"].diff().dt.total_seconds()
)

# --- time since session start ---
session_start_map = ALL_EVENTS_A.groupby("session_id")["timestamp"].min()
CANDIDATES["time_since_session_start_s"] = (
    CANDIDATES["timestamp"] - CANDIDATES["session_id"].map(session_start_map)
).dt.total_seconds()

# --- did the actual app/window title change vs the previous candidate? ---
CANDIDATES["prev_app_name"] = CANDIDATES.groupby("session_id")["active_app_name"].shift(1)
CANDIDATES["prev_window_title"] = CANDIDATES.groupby("session_id")["active_window_title"].shift(1)
CANDIDATES["app_changed"] = (CANDIDATES["active_app_name"] != CANDIDATES["prev_app_name"]).astype(int)
CANDIDATES["title_changed"] = (CANDIDATES["active_window_title"] != CANDIDATES["prev_window_title"]).astype(int)

# --- one-hot style flags for the restricted event types ---
for event_type_name in sorted(BOUNDARY_EVENT_TYPES):
    CANDIDATES[f"is_{event_type_name}"] = (CANDIDATES["event_type"] == event_type_name).astype(int)


def count_events_in_window(session_events_ns, target_ns, window_ms, direction):
    if direction == "prior":
        lo = target_ns - window_ms * 1_000_000
        left = np.searchsorted(session_events_ns, lo, side="left")
        right = np.searchsorted(session_events_ns, target_ns, side="left")
    else:
        hi = target_ns + window_ms * 1_000_000
        left = np.searchsorted(session_events_ns, target_ns, side="right")
        right = np.searchsorted(session_events_ns, hi, side="right")
    return right - left


def add_window_counts(candidates_df, events_df, window_ms, direction, colname):
    counts = np.zeros(len(candidates_df), dtype=int)
    candidates_df = candidates_df.reset_index(drop=True)
    for session_id, cand_group in candidates_df.groupby("session_id"):
        session_events_ns = (
            events_df.loc[events_df["session_id"] == session_id, "timestamp"]
            .sort_values().values.astype("datetime64[ns]").astype("int64")
        )
        target_ns = cand_group["timestamp"].values.astype("datetime64[ns]").astype("int64")
        counts[cand_group.index.values] = count_events_in_window(
            session_events_ns, target_ns, window_ms, direction
        )
    candidates_df[colname] = counts
    return candidates_df


##local activity density: how busy was the user just before/after this candidate ---
CANDIDATES = add_window_counts(CANDIDATES, ALL_EVENTS_A, 10000, "prior", "events_in_prior_10s")
CANDIDATES = add_window_counts(CANDIDATES, ALL_EVENTS_A, 10000, "next", "events_in_next_10s")

print("Candidate feature table shape:", CANDIDATES.shape)
print("\nColumns:", sorted(CANDIDATES.columns.tolist()))


Candidate feature table shape: (53109, 26)

Columns: ['active_app_name', 'active_window_title', 'app_changed', 'event_id', 'event_type', 'events_in_next_10s', 'events_in_prior_10s', 'gap_before_candidate_s', 'is_app_switch', 'is_browser_navigation', 'is_browser_tab_event', 'is_dialog_closed', 'is_dialog_opened', 'is_window_state_change', 'is_window_title_change', 'layer', 'prev_app_name', 'prev_window_title', 'sequence_number', 'session_id', 'source_chunk_dir', 'time_since_prev_event_s', 'time_since_session_start_s', 'timestamp', 'timestamp_iso', 'title_changed']


## Label candidates against ground truth (resume-aware)

Labels come from `GT_SEGMENTS_A["start"]`, which already includes both
`process_started` and `process_resumed` origins (unlike the original
`segmentation_model.ipynb`, which only used `process_started`).

In [9]:
LABEL_TOLERANCE_S = 2.0

CANDIDATES["boundary_label"] = 0

for session_id, gt_group in GT_SEGMENTS_A.groupby("session_id"):
    session_candidate_idx = CANDIDATES.index[CANDIDATES["session_id"] == session_id]
    if len(session_candidate_idx) == 0:
        continue
    candidate_times = CANDIDATES.loc[session_candidate_idx, "timestamp"]
    for gt_start in gt_group["start"]:
        distances = (candidate_times - gt_start).abs().dt.total_seconds()
        nearest_local_idx = distances.idxmin()
        if distances.loc[nearest_local_idx] <= LABEL_TOLERANCE_S:
            CANDIDATES.loc[nearest_local_idx, "boundary_label"] = 1

n_positive = int(CANDIDATES["boundary_label"].sum())
print(f"Positive boundary labels: {n_positive} / {len(GT_SEGMENTS_A)} true segment starts")
print(f"Positive rate among candidates: {CANDIDATES["boundary_label"].mean():.3%}")
print(f"(Original notebook would have produced at most 1,819 positives here -- "
      f"this version can reach up to {len(GT_SEGMENTS_A)} because process_resumed is included.)")


Positive boundary labels: 1691 / 2009 true segment starts
Positive rate among candidates: 3.184%
(Original notebook would have produced at most 1,819 positives here -- this version can reach up to 2009 because process_resumed is included.)


## Session-level train / validation / test split (70 / 15 / 15)

In [10]:
session_ids = CANDIDATES["session_id"].drop_duplicates().tolist()
rng = random.Random(RANDOM_SEED)
rng.shuffle(session_ids)

n_sessions = len(session_ids)
n_train = int(n_sessions * 0.70)
n_val = int(n_sessions * 0.15)

train_sessions = set(session_ids[:n_train])
val_sessions = set(session_ids[n_train:n_train + n_val])
test_sessions = set(session_ids[n_train + n_val:])

train_data = CANDIDATES[CANDIDATES["session_id"].isin(train_sessions)].copy()
val_data = CANDIDATES[CANDIDATES["session_id"].isin(val_sessions)].copy()
test_data = CANDIDATES[CANDIDATES["session_id"].isin(test_sessions)].copy()

print(f"Sessions  -> train:{len(train_sessions)}  val:{len(val_sessions)}  test:{len(test_sessions)}")
print(f"Rows      -> train:{len(train_data):,}  val:{len(val_data):,}  test:{len(test_data):,}")
print(f"Positives -> train:{train_data["boundary_label"].sum()}  "
      f"val:{val_data["boundary_label"].sum()}  test:{test_data["boundary_label"].sum()}")


Sessions  -> train:44  val:9  test:10
Rows      -> train:37,858  val:6,392  test:8,859
Positives -> train:1204  val:194  test:293


##  Feature preprocessing

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAL_FEATURES = ["event_type", "layer"]
NUMERIC_FEATURES = [
    "gap_before_candidate_s", "time_since_prev_event_s", "time_since_session_start_s",
    "app_changed", "title_changed", "events_in_prior_10s", "events_in_next_10s",
] + [f"is_{event_type_name}" for event_type_name in sorted(BOUNDARY_EVENT_TYPES)]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

for df in (train_data, val_data, test_data):
    df[NUMERIC_FEATURES] = df[NUMERIC_FEATURES].fillna(0)

X_train, y_train = train_data[FEATURE_COLUMNS], train_data["boundary_label"]
X_val, y_val = val_data[FEATURE_COLUMNS], val_data["boundary_label"]
X_test, y_test = test_data[FEATURE_COLUMNS], test_data["boundary_label"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ("numeric", StandardScaler(), NUMERIC_FEATURES),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print("Processed feature matrix shape (train):", X_train_proc.shape)


Processed feature matrix shape (train): (37858, 23)


## Train candidate models

Four models, deliberately: Logistic Regression (interpretable baseline),
Random Forest and XGBoost (the two classical non-linear models we
discussed), and CatBoost (handles the categorical `event_type`/`layer`
columns natively, worth comparing since it was already part of your
environment). AdaBoost and an MLP/ANN are intentionally **not** included
here: AdaBoost has not shown meaningful lift over RF/XGBoost on this kind
of tabular problem, and a neural network is not justified on this data
size for the reasons discussed earlier in the project (see work log) --
adding either would be complexity without a demonstrated reason.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

positive_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print("Class imbalance ratio (negative:positive):", round(positive_weight, 1), ": 1")

MODELS = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", max_depth=15,
        min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=positive_weight,
        eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05, loss_function="Logloss",
        auto_class_weights="Balanced", random_state=RANDOM_SEED, verbose=False
    ),
}

for name, model in MODELS.items():
    _t0 = time.perf_counter()
    model.fit(X_train_proc, y_train)
    print(f"{name}: trained in {time.perf_counter() - _t0:.1f}s")


Class imbalance ratio (negative:positive): 30.4 : 1
Logistic Regression: trained in 0.1s
Random Forest: trained in 1.2s
XGBoost: trained in 0.7s
CatBoost: trained in 2.3s


## Validation threshold sweep, select best model + threshold

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

THRESHOLDS = np.arange(0.10, 0.96, 0.05)
sweep_results = []
val_probabilities = {}

for name, model in MODELS.items():
    probabilities = model.predict_proba(X_val_proc)[:, 1]
    val_probabilities[name] = probabilities
    for threshold in THRESHOLDS:
        predictions = (probabilities >= threshold).astype(int)
        sweep_results.append({
            "model": name,
            "threshold": round(float(threshold), 2),
            "precision": precision_score(y_val, predictions, zero_division=0),
            "recall": recall_score(y_val, predictions, zero_division=0),
            "f1": f1_score(y_val, predictions, zero_division=0),
            "n_predicted": int(predictions.sum()),
        })

sweep_df = pd.DataFrame(sweep_results)
best_per_model = (
    sweep_df.sort_values(["model", "f1"], ascending=[True, False])
    .groupby("model").head(1)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("Best threshold per model (by validation F1):")
print(best_per_model.to_string(index=False))

BEST_MODEL_NAME = best_per_model.iloc[0]["model"]
BEST_THRESHOLD = float(best_per_model.iloc[0]["threshold"])
print(f"\nSelected model: {BEST_MODEL_NAME}  @ threshold {BEST_THRESHOLD}")


Best threshold per model (by validation F1):
              model  threshold  precision   recall       f1  n_predicted
           CatBoost       0.95   0.953125 0.943299 0.948187          192
            XGBoost       0.95   0.957895 0.938144 0.947917          190
      Random Forest       0.90   0.967742 0.927835 0.947368          186
Logistic Regression       0.80   0.205514 0.422680 0.276560          399

Selected model: CatBoost  @ threshold 0.95


In [ ]:

# The single train/val split gave CatBoost a threshold (0.95) that looked
# excellent on validation (~94% recall) but only recovered ~75% of
# candidate-matched true starts on test. With only ~9 validation sessions,
# that single split is too small to trust for picking a threshold. This
# re-estimates the threshold using 5-fold GROUP cross-validation across
# train+validation combined (grouped by session, so no session ever
# appears in both the fit and holdout side of a fold), giving a much
# larger, more stable out-of-fold sample to sweep thresholds against.
# Test data is NOT touched here -- only train+val.
import pandas as pd
from sklearn.model_selection import GroupKFold

combined_pool = pd.concat([train_data, val_data], ignore_index=True)
X_pool = combined_pool[FEATURE_COLUMNS]
y_pool = combined_pool["boundary_label"]
groups_pool = combined_pool["session_id"]

# Note: reusing the preprocessor already fit on train_data only (from Cell 9)
# to transform the combined pool is a small simplification -- ideally the
# preprocessor would be refit per fold. Since it's a monotonic scale/encode
# transform (not something that can leak label information), this shouldn't
# materially change the threshold chosen, only save time.
X_pool_proc = preprocessor.transform(X_pool)


def build_fresh_model(name):
    if name == "Logistic Regression":
        return LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED)
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=300, class_weight="balanced", max_depth=15,
                                       min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_SEED)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, scale_pos_weight=positive_weight,
                              eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1)
    if name == "CatBoost":
        return CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, loss_function="Logloss",
                                   auto_class_weights="Balanced", random_state=RANDOM_SEED, verbose=False)
    raise ValueError(name)


N_SPLITS = 5
group_kfold = GroupKFold(n_splits=N_SPLITS)
oof_probabilities = np.zeros(len(combined_pool))

for fold_idx, (fit_idx, holdout_idx) in enumerate(group_kfold.split(X_pool_proc, y_pool, groups=groups_pool)):
    fold_model = build_fresh_model(BEST_MODEL_NAME)
    fold_model.fit(X_pool_proc[fit_idx], y_pool.iloc[fit_idx])
    oof_probabilities[holdout_idx] = fold_model.predict_proba(X_pool_proc[holdout_idx])[:, 1]
    print(f"Fold {fold_idx + 1}/{N_SPLITS} done")

oof_results = []
for threshold in THRESHOLDS:
    preds = (oof_probabilities >= threshold).astype(int)
    oof_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_pool, preds, zero_division=0),
        "recall": recall_score(y_pool, preds, zero_division=0),
        "f1": f1_score(y_pool, preds, zero_division=0),
    })

oof_df = pd.DataFrame(oof_results).sort_values("f1", ascending=False)
print("\nTop cross-validated thresholds by out-of-fold F1:")
print(oof_df.head(10).to_string(index=False))

OLD_THRESHOLD = BEST_THRESHOLD
BEST_THRESHOLD = float(oof_df.iloc[0]["threshold"])

print(f"\nOriginal single-split threshold : {OLD_THRESHOLD}")
print(f"Cross-validated threshold (new) : {BEST_THRESHOLD}")
print("\n-> Re-run Cell 12 and Cell 13 now so the test evaluation uses this corrected threshold.")

Fold 1/5 done
Fold 2/5 done
Fold 3/5 done
Fold 4/5 done
Fold 5/5 done

Top cross-validated thresholds by out-of-fold F1:
 threshold  precision   recall       f1
      0.95   0.918750 0.841202 0.878267
      0.90   0.874182 0.859800 0.866931
      0.85   0.829615 0.877682 0.852972
      0.80   0.796023 0.887697 0.839364
      0.75   0.762568 0.900572 0.825845
      0.70   0.728687 0.904864 0.807275
      0.65   0.703315 0.910587 0.793641
      0.60   0.674051 0.914163 0.775956
      0.55   0.651316 0.920601 0.762893
      0.50   0.626384 0.930615 0.748777

Original single-split threshold : 0.95
Cross-validated threshold (new) : 0.95

-> Re-run Cell 12 and Cell 13 now so the test evaluation uses this corrected threshold.


## Apply best model to the TEST set + duration-based post-processing

The 15-second minimum-duration merge is applied here, not the arbitrary
5-second gap value used in the original notebook -- 15s sits just under
the empirically measured true-segment minimum (16.64s), so it should
remove short false-positive segments without touching any real ones.

In [15]:
best_model = MODELS[BEST_MODEL_NAME]
test_probabilities = best_model.predict_proba(X_test_proc)[:, 1]

test_data = test_data.copy()
test_data["boundary_probability"] = test_probabilities
test_data["boundary_prediction"] = (test_probabilities >= BEST_THRESHOLD).astype(int)

print("Predicted boundaries on test set (pre-merge):", int(test_data["boundary_prediction"].sum()))

MIN_SEGMENT_DURATION_S = 15  # evidence-based: true GT segment minimum is 16.64s


def merge_short_segments(boundary_timestamps, min_duration_s):
    if len(boundary_timestamps) == 0:
        return []
    kept = [boundary_timestamps[0]]
    for ts in boundary_timestamps[1:]:
        if (ts - kept[-1]).total_seconds() >= min_duration_s:
            kept.append(ts)
    return kept


def boundaries_to_segments(boundary_ts, session_id, session_end_ts):
    starts = list(boundary_ts)
    ends = starts[1:] + [session_end_ts]
    return pd.DataFrame({"session_id": session_id, "pred_start": starts, "pred_end": ends})


predicted_segments_list = []
for session_id, group in test_data[test_data["boundary_prediction"] == 1].groupby("session_id"):
    session_end_ts = ALL_EVENTS_A.loc[ALL_EVENTS_A["session_id"] == session_id, "timestamp"].max()
    boundary_ts = sorted(group["timestamp"].tolist())
    merged_ts = merge_short_segments(boundary_ts, MIN_SEGMENT_DURATION_S)
    if merged_ts:
        predicted_segments_list.append(boundaries_to_segments(merged_ts, session_id, session_end_ts))

PREDICTED_SEGMENTS_TEST = (
    pd.concat(predicted_segments_list, ignore_index=True)
    if predicted_segments_list else
    pd.DataFrame(columns=["session_id", "pred_start", "pred_end"])
)

print("Predicted segments on test set (post-merge):", len(PREDICTED_SEGMENTS_TEST))


Predicted boundaries on test set (pre-merge): 243
Predicted segments on test set (post-merge): 234


## Evaluate on held-out test sessions: boundary metrics AND segment IoU

This is the evaluation the original notebook was missing: segment-level
matching (via interval IoU), not just point-in-time boundary matching.

In [16]:
def evaluate_boundaries(true_by_session, pred_by_session, tolerance_ms):
    true_positive, false_negative, matched_pred, total_pred = 0, 0, 0, 0
    for session_id, true_ts in true_by_session.items():
        pred_ts = pred_by_session.get(session_id, np.array([]))
        total_pred += len(pred_ts)
        if len(pred_ts) == 0:
            false_negative += len(true_ts)
            continue
        pred_ns = np.sort(pred_ts.astype("int64"))
        true_ns = true_ts.astype("int64")
        gaps = nearest_gap_ms(pred_ns, true_ns)
        hits = gaps <= tolerance_ms
        true_positive += hits.sum()
        false_negative += (~hits).sum()
        gaps_rev = nearest_gap_ms(np.sort(true_ns), pred_ns)
        matched_pred += (gaps_rev <= tolerance_ms).sum()

    precision = matched_pred / total_pred if total_pred else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1


test_gt = GT_SEGMENTS_A[GT_SEGMENTS_A["session_id"].isin(test_sessions)]
true_by_session_test = {
    sid: g["start"].values.astype("datetime64[ns]") for sid, g in test_gt.groupby("session_id")
}
pred_by_session_test = {
    sid: g["pred_start"].values.astype("datetime64[ns]") for sid, g in PREDICTED_SEGMENTS_TEST.groupby("session_id")
}

print("--- Boundary-level evaluation (test sessions only) ---")
boundary_metrics_by_tolerance = {}
for tolerance_ms in [1000, 2000, 5000]:
    precision, recall, f1 = evaluate_boundaries(true_by_session_test, pred_by_session_test, tolerance_ms)
    boundary_metrics_by_tolerance[tolerance_ms] = (precision, recall, f1)
    print(f"@ {tolerance_ms}ms -> precision={precision:.1%}  recall={recall:.1%}  f1={f1:.1%}")


def interval_iou(a_start_ns, a_end_ns, b_start_ns, b_end_ns):
    latest_start = max(a_start_ns, b_start_ns)
    earliest_end = min(a_end_ns, b_end_ns)
    intersection = max(0, earliest_end - latest_start)
    union = (a_end_ns - a_start_ns) + (b_end_ns - b_start_ns) - intersection
    return intersection / union if union > 0 else 0.0


def evaluate_segments_iou(true_segments_df, pred_segments_df, iou_threshold=0.5):
    matched_true_total = 0
    matched_pred_total = 0
    total_true = len(true_segments_df)
    total_pred = len(pred_segments_df)

    for session_id in true_segments_df["session_id"].unique():
        true_rows = true_segments_df[true_segments_df["session_id"] == session_id].reset_index(drop=True)
        pred_rows = pred_segments_df[pred_segments_df["session_id"] == session_id].reset_index(drop=True)
        used_pred_idx = set()
        for _, true_row in true_rows.iterrows():
            best_iou, best_idx = 0.0, None
            for pred_idx, pred_row in pred_rows.iterrows():
                if pred_idx in used_pred_idx:
                    continue
                score = interval_iou(
                    true_row["start"].value, true_row["end"].value,
                    pred_row["pred_start"].value, pred_row["pred_end"].value,
                )
                if score > best_iou:
                    best_iou, best_idx = score, pred_idx
            if best_idx is not None and best_iou >= iou_threshold:
                matched_true_total += 1
                used_pred_idx.add(best_idx)
        matched_pred_total += len(used_pred_idx)

    precision = matched_pred_total / total_pred if total_pred else 0
    recall = matched_true_total / total_true if total_true else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1, matched_true_total, total_true, total_pred


print("\n--- Segment-level IoU evaluation (test sessions only) ---")
iou_precision, iou_recall, iou_f1, n_matched, n_true, n_pred = evaluate_segments_iou(
    test_gt, PREDICTED_SEGMENTS_TEST, iou_threshold=0.5
)
print(f"IoU>=0.5 -> precision={iou_precision:.1%}  recall={iou_recall:.1%}  f1={iou_f1:.1%}")
print(f"Matched {n_matched} / {n_true} true segments; {n_pred} predicted segments total")


--- Boundary-level evaluation (test sessions only) ---
@ 1000ms -> precision=78.2%  recall=56.0%  f1=65.2%
@ 2000ms -> precision=95.7%  recall=68.5%  f1=79.9%
@ 5000ms -> precision=96.2%  recall=68.8%  f1=80.2%

--- Segment-level IoU evaluation (test sessions only) ---
IoU>=0.5 -> precision=87.2%  recall=62.4%  f1=72.7%
Matched 204 / 327 true segments; 234 predicted segments total


In [ ]:


miss_records = []
for session_id, gt_group in test_gt.groupby("session_id"):
    session_candidates = test_data[test_data["session_id"] == session_id].sort_values("timestamp").reset_index(drop=True)
    if session_candidates.empty:
        continue
    cand_times = session_candidates["timestamp"]
    n_missed = 0
    n_total_with_candidate = 0
    for gt_start in gt_group["start"]:
        distances = (cand_times - gt_start).abs().dt.total_seconds()
        nearest_idx = distances.idxmin()
        if distances.loc[nearest_idx] > 2.0:
            continue
        n_total_with_candidate += 1
        if session_candidates.loc[nearest_idx, "boundary_prediction"] == 0:
            n_missed += 1
    if n_total_with_candidate > 0:
        miss_records.append({
            "session_id": session_id,
            "n_missed": n_missed,
            "n_total_with_candidate": n_total_with_candidate,
            "miss_rate": n_missed / n_total_with_candidate,
        })

miss_df = pd.DataFrame(miss_records).sort_values("miss_rate", ascending=False)
print(miss_df.to_string(index=False))

                         session_id  n_missed  n_total_with_candidate  miss_rate
ses_20260701-035622-SIDDHIGUPTAB00B        17                      34   0.500000
ses_20260701-092027-SIDDHIGUPTAB00B        12                      30   0.400000
  ses_20260701-032825-CHAITANYA0BCF         7                      27   0.259259
  ses_20260701-051820-CHAITANYA0BCF         6                      25   0.240000
ses_20260630-163424-SIDDHIGUPTAB00B         6                      28   0.214286
  ses_20260630-135451-CHAITANYA0BCF         6                      30   0.200000
            ses_20260701-145907-MSI         5                      28   0.178571
         ses_20260701-110917-Marcos         4                      27   0.148148
  ses_20260630-124826-CHAITANYA0BCF         4                      30   0.133333
         ses_20260630-145433-JAYESH         3                      34   0.088235


## Retrain on train+validation, save final artifacts

Only now -- after the held-out test evaluation above -- do we retrain on
train+validation combined and persist the model. This mirrors your
original notebook's artifact layout so nothing downstream needs to change
paths.

In [17]:
import joblib

final_train_data = pd.concat([train_data, val_data], ignore_index=True)
X_final_train = final_train_data[FEATURE_COLUMNS]
y_final_train = final_train_data["boundary_label"]

final_preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ("numeric", StandardScaler(), NUMERIC_FEATURES),
])
X_final_train_proc = final_preprocessor.fit_transform(X_final_train)

FINAL_MODEL_BUILDERS = {
    "Logistic Regression": lambda: LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED
    ),
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=300, class_weight="balanced", max_depth=15,
        min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_SEED
    ),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=positive_weight,
        eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
    ),
    "CatBoost": lambda: CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05, loss_function="Logloss",
        auto_class_weights="Balanced", random_state=RANDOM_SEED, verbose=False
    ),
}

final_model = FINAL_MODEL_BUILDERS[BEST_MODEL_NAME]()
final_model.fit(X_final_train_proc, y_final_train)

model_path = MODELS_DIR / "best_segmentation_model.pkl"
preprocessor_path = PREPROCESSING_DIR / "best_preprocessor.pkl"

joblib.dump(final_model, model_path)
joblib.dump(final_preprocessor, preprocessor_path)

precision_2000, recall_2000, f1_2000 = evaluate_boundaries(true_by_session_test, pred_by_session_test, 2000)

metadata = {
    "model_name": BEST_MODEL_NAME,
    "model_version": "V2_context_shift_candidates_resume_aware",
    "random_seed": RANDOM_SEED,
    "boundary_threshold": BEST_THRESHOLD,
    "min_segment_duration_seconds": MIN_SEGMENT_DURATION_S,
    "boundary_event_types_used_as_candidates": sorted(BOUNDARY_EVENT_TYPES),
    "feature_columns": FEATURE_COLUMNS,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "gt_boundary_sources": ["process_started", "process_resumed"],
    "label_tolerance_seconds": LABEL_TOLERANCE_S,
    "train_sessions": len(train_sessions),
    "validation_sessions": len(val_sessions),
    "test_sessions": len(test_sessions),
    "test_boundary_precision_2000ms": precision_2000,
    "test_boundary_recall_2000ms": recall_2000,
    "test_boundary_f1_2000ms": f1_2000,
    "test_segment_iou50_precision": iou_precision,
    "test_segment_iou50_recall": iou_recall,
    "test_segment_iou50_f1": iou_f1,
    "notes": (
        "V2 differs from the original segmentation_model.ipynb in three ways: "
        "(1) GT boundaries include process_resumed, not just process_started; "
        "(2) candidates are restricted to context-shift event types instead of "
        "ALL raw events; (3) minimum segment duration for post-processing is set "
        "to 15s based on the empirically measured true-segment minimum duration "
        "(16.64s), not an arbitrary fixed gap value. Segment LABELING (assigning "
        "a process identity to each segment) is intentionally out of scope for "
        "this model -- see the separate labeling notebook."
    ),
}

metadata_path = METADATA_DIR / "segmentation_model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved model to       :", model_path)
print("Saved preprocessor to:", preprocessor_path)
print("Saved metadata to    :", metadata_path)


Saved model to       : c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\models\best_segmentation_model.pkl
Saved preprocessor to: c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\preprocessing\best_preprocessor.pkl
Saved metadata to    : c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\metadata\segmentation_model_metadata.json


##  Verify saved artifacts by reloading them

In [18]:
loaded_model = joblib.load(model_path)
loaded_preprocessor = joblib.load(preprocessor_path)

print("Reloaded model type       :", type(loaded_model).__name__)
print("Reloaded preprocessor type:", type(loaded_preprocessor).__name__)

with open(metadata_path, "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

print("\nSaved metadata:")
print(json.dumps(saved_metadata, indent=2))


Reloaded model type       : CatBoostClassifier
Reloaded preprocessor type: ColumnTransformer

Saved metadata:
{
  "model_name": "CatBoost",
  "model_version": "V2_context_shift_candidates_resume_aware",
  "random_seed": 42,
  "boundary_threshold": 0.95,
  "min_segment_duration_seconds": 15,
  "boundary_event_types_used_as_candidates": [
    "app_switch",
    "browser_navigation",
    "browser_tab_event",
    "dialog_closed",
    "dialog_opened",
    "window_state_change",
    "window_title_change"
  ],
  "feature_columns": [
    "event_type",
    "layer",
    "gap_before_candidate_s",
    "time_since_prev_event_s",
    "time_since_session_start_s",
    "app_changed",
    "title_changed",
    "events_in_prior_10s",
    "events_in_next_10s",
    "is_app_switch",
    "is_browser_navigation",
    "is_browser_tab_event",
    "is_dialog_closed",
    "is_dialog_opened",
    "is_window_state_change",
    "is_window_title_change"
  ],
  "categorical_features": [
    "event_type",
    "layer"
  

In [ ]:


# 1. Candidate recall computed ONLY on test sessions (not all of Dataset A)
test_candidates_only = CANDIDATES[CANDIDATES["session_id"].isin(test_sessions)]
test_gt_gap_ms = compute_nearest_gaps(test_gt, test_candidates_only)
print("Candidate recall @ 2000ms, TEST SESSIONS ONLY:", f"{(test_gt_gap_ms <= 2000).mean():.1%}")
print("(Compare to 84.2% on the full Dataset A from Cell 5 -- if this is notably lower,")
print(" the recall drop is mostly a sampling/coverage issue, not a modeling issue.)")

# 2. For each true test-set start, classify WHY it was missed (or wasn't)
pre_merge_boundaries_by_session = {
    sid: sorted(g.loc[g["boundary_prediction"] == 1, "timestamp"].tolist())
    for sid, g in test_data.groupby("session_id")
}
post_merge_boundaries_by_session = {
    sid: sorted(g["pred_start"].tolist())
    for sid, g in PREDICTED_SEGMENTS_TEST.groupby("session_id")
}

categories = []
for session_id, gt_group in test_gt.groupby("session_id"):
    session_candidates = test_data[test_data["session_id"] == session_id].sort_values("timestamp").reset_index(drop=True)
    if session_candidates.empty:
        categories.extend(["no_candidate"] * len(gt_group))
        continue

    cand_times = session_candidates["timestamp"]
    pre_merge_set = set(pre_merge_boundaries_by_session.get(session_id, []))
    post_merge_set = set(post_merge_boundaries_by_session.get(session_id, []))

    for gt_start in gt_group["start"]:
        distances = (cand_times - gt_start).abs().dt.total_seconds()
        nearest_idx = distances.idxmin()
        if distances.loc[nearest_idx] > 2.0:
            categories.append("no_candidate_nearby")
            continue
        nearest_row = session_candidates.loc[nearest_idx]
        if nearest_row["boundary_prediction"] == 0:
            categories.append("model_missed")
        elif nearest_row["timestamp"] in post_merge_set:
            categories.append("recovered")
        else:
            categories.append("merged_away")

category_counts = pd.Series(categories).value_counts()
print("\nWhy each true test-set start was recovered or missed:")
print(category_counts.to_string())
print("\nAs a fraction of all", len(categories), "true test-set starts:")
print((category_counts / len(categories)).round(3).to_string())

Candidate recall @ 2000ms, TEST SESSIONS ONLY: 89.6%
(Compare to 84.2% on the full Dataset A from Cell 5 -- if this is notably lower,
 the recall drop is mostly a sampling/coverage issue, not a modeling issue.)

Why each true test-set start was recovered or missed:
recovered              215
model_missed            70
no_candidate_nearby     34
merged_away              8

As a fraction of all 327 true test-set starts:
recovered              0.657
model_missed           0.214
no_candidate_nearby    0.104
merged_away            0.024
